# Пункт 3. Скорость умножения на веса INT4

В пункте 1 мы меняли масштаб при округлении весов. Здесь проверим, влияет ли этот выбор на скорость умножения. Одну и ту же матрицу весов сожмём двумя способами. В `absmax` берём максимальный модуль веса в строке и делим на 7. В `mse` пробуем 18 масштабов и выбираем тот, который даёт наименьшую ошибку восстановления строки. Обе версии сравним с обычным `torch.mm` на плотных весах.

В Colab выберите GPU и нажмите **Выполнить все**. Веса генерируются прямо в ноутбуке. От Llama-3.2-1B-Instruct нам нужны только размеры матриц. [Конфигурация модели](https://huggingface.co/unsloth/Llama-3.2-1B-Instruct/blob/main/config.json)

| Проекции Llama | Выходные признаки N | Входные признаки K |
|---|---:|---:|
| Q / O | 2048 | 2048 |
| K / V | 512 | 2048 |
| Gate / Up | 8192 | 2048 |
| Down | 2048 | 8192 |
| Выходная голова | 128256 | 2048 |

Для каждой строки таблицы возьмём по 128, 512 и 2048 токенов. Всего получится 15 замеров для каждого способа умножения.

Исходные веса для квантизации всегда в FP16. На T4 и GTX 1660 Ti активации и плотные веса тоже в FP16, а INT4-кернель использует инструкцию `mma.sync`. На Ampere и новее ноутбук берёт активации и плотные веса в BF16 и вызывает `tl.dot` для INT4. Чтобы попробовать `tl.dot` на T4, поменяйте `T4_KERNEL` на `"tl.dot"` в ячейке с проверкой упаковки.

In [1]:
import torch

if not torch.cuda.is_available() or torch.version.cuda is None:
    raise RuntimeError("В Colab выберите «Среда выполнения → Сменить среду выполнения → GPU T4».")
device = torch.cuda.get_device_properties(torch.cuda.current_device())
x_dtype = torch.bfloat16 if device.major >= 8 else torch.float16
dtype_label = "BF16" if x_dtype == torch.bfloat16 else "FP16"
dense_column = f"{dtype_label.lower()}_ms"

import triton

runtime = {
    "gpu": device.name,
    "compute_capability": f"{device.major}.{device.minor}",
    "torch_version": str(torch.__version__),
    "cuda_version": torch.version.cuda,
    "triton_version": triton.__version__,
}
print(f"{device.name} · {dtype_label} × INT4 и {dtype_label} × {dtype_label}")
print(f"PyTorch {torch.__version__} · CUDA {torch.version.cuda} · Triton {triton.__version__}")

NVIDIA GeForce GTX 1660 Ti · FP16 × INT4 и FP16 × FP16
PyTorch 2.10.0+cu126 · CUDA 12.6 · Triton 3.8.0


## Подготовка кернелей

Следующие ячейки запишут код из пунктов 1 и 2 в файлы, из которых Triton сможет его скомпилировать. Квантизатор тот же, что в `1_int4-quantization/quantize_int4.py`. Числа у `absmax` и `mse` могут получиться разными, но упакованы они одинаково — два веса в байте и один масштаб FP32 на строку. Поэтому обе матрицы можно передать одной функции умножения.

In [2]:
%%writefile task3_quantizer.py
"""GPU quantization of FP16/BF16 weight matrices into signed packed INT4."""

import torch
import triton
import triton.language as tl


@triton.jit
def row_max_kernel(X, RowMax, N: tl.constexpr, BLOCK_N: tl.constexpr):
    row = tl.program_id(0)
    cols = tl.arange(0, BLOCK_N)
    row_max = tl.full((), 0.0, tl.float32)
    for start in range(tl.cdiv(N, BLOCK_N)):
        j = start * BLOCK_N + cols
        x = tl.load(X + row * N + j, mask=j < N, other=0.0).to(tl.float32)
        row_max = tl.maximum(row_max, tl.max(tl.abs(x), 0))
    tl.store(RowMax + row, row_max)


@triton.jit
def quantize_pack_kernel(X, RowMax, Packed, Scales,
                         N: tl.constexpr, BLOCK: tl.constexpr):
    row = tl.program_id(0)
    block = tl.program_id(1)
    max_val = tl.load(RowMax + row)

    scale = tl.where(max_val > 0.0, tl.div_rn(max_val, 7.0), 1.0)
    if block == 0:
        tl.store(Scales + row, scale)

    j = block * BLOCK + tl.arange(0, BLOCK)
    even = 2 * j
    odd = even + 1
    x0 = tl.load(X + row * N + even, mask=even < N, other=0.0).to(tl.float32)
    x1 = tl.load(X + row * N + odd, mask=odd < N, other=0.0).to(tl.float32)

    a0 = tl.abs(x0) * 14.0
    a1 = tl.abs(x1) * 14.0
    q0_abs = tl.full((BLOCK,), 0, tl.int32)
    q1_abs = tl.full((BLOCK,), 0, tl.int32)
    for i in tl.static_range(0, 7):
        threshold = max_val * (2 * i + 1)
        q0_abs += ((max_val > 0.0) & (a0 >= threshold)).to(tl.int32)
        q1_abs += ((max_val > 0.0) & (a1 >= threshold)).to(tl.int32)

    q0 = tl.where(x0 < 0.0, -q0_abs, q0_abs)
    q1 = tl.where(x1 < 0.0, -q1_abs, q1_abs)
    byte = ((q0 & 15) | ((q1 & 15) << 4)).to(tl.int8)
    tl.store(Packed + row * (N // 2) + j, byte, mask=j < N // 2)


@triton.jit
def fused_quantize_pack_kernel(X, Packed, Scales,
                               N: tl.constexpr, BLOCK_N: tl.constexpr):
    row = tl.program_id(0)
    j = tl.arange(0, BLOCK_N)
    x = tl.load(X + row * N + j, mask=j < N, other=0.0).to(tl.float32)
    max_val = tl.max(tl.abs(x), 0)
    scale = tl.where(max_val > 0.0, tl.div_rn(max_val, 7.0), 1.0)
    tl.store(Scales + row, scale)

    # Квантизация выполняется над всеми элементами строки (пороговая
    # арифметика как в baseline), затем упаковываются пары соседей.
    a = tl.abs(x) * 14.0
    q_abs = tl.full((BLOCK_N,), 0, tl.int32)
    for k in tl.static_range(0, 7):
        threshold = max_val * (2 * k + 1)
        q_abs += ((max_val > 0.0) & (a >= threshold)).to(tl.int32)
    q = tl.where(x < 0.0, -q_abs, q_abs)

    # Gather по четным и нечетным позициям; по одной паре на байт.
    pairs = tl.arange(0, BLOCK_N // 2)
    q0 = tl.gather(q, 2 * pairs, axis=0)
    q1 = tl.gather(q, 2 * pairs + 1, axis=0)
    packed = ((q0 & 15) | ((q1 & 15) << 4)).to(tl.int8)
    tl.store(Packed + row * (N // 2) + pairs, packed, mask=pairs < N // 2)


RATIOS = (1.0, 0.99, 0.975, 0.95, 0.925, 0.90, 0.875, 0.85, 0.825,
          0.80, 0.775, 0.75, 0.725, 0.70, 0.65, 0.60, 0.55, 0.50)


@triton.jit
def _abs_code(x, clip):
    # Same threshold comparisons as task 1, including ties and the zero row.
    a14 = tl.abs(x) * 14.0
    code = tl.full(x.shape, 0, tl.int32)
    for i in tl.static_range(7):
        code += (a14 >= clip * (2 * i + 1)).to(tl.int32)
    return tl.where(clip > 0.0, code, 0)


@triton.jit
def _scale(clip):
    return tl.where(clip > 0.0, tl.div_rn(clip, 7.0), 1.0)


@triton.jit
def _row_max(W, Maxima, K: tl.constexpr, BLOCK: tl.constexpr):
    row = tl.program_id(0)
    lane = tl.arange(0, BLOCK)
    maximum = tl.full((BLOCK,), 0.0, tl.float32)
    for tile in range(tl.cdiv(K, BLOCK)):
        col = tile * BLOCK + lane
        x = tl.load(W + row * K + col, col < K, other=0.0).to(tl.float32)
        maximum = tl.maximum(maximum, tl.abs(x))
    tl.store(Maxima + row, tl.max(maximum, 0))


@triton.jit
def _score(W, Maxima, Ratios, Scores, K: tl.constexpr, COUNT: tl.constexpr, BLOCK: tl.constexpr):
    row, candidate = tl.program_id(0), tl.program_id(1)
    clip = tl.load(Maxima + row) * tl.load(Ratios + candidate)
    scale = _scale(clip)
    lane = tl.arange(0, BLOCK)
    sums = tl.full((BLOCK,), 0.0, tl.float64)
    for tile in range(tl.cdiv(K, BLOCK)):
        col = tile * BLOCK + lane
        x = tl.load(W + row * K + col, col < K, other=0.0).to(tl.float32)
        reconstructed_magnitude = _abs_code(x, clip).to(tl.float32) * scale
        error = tl.abs(x).to(tl.float64) - reconstructed_magnitude.to(tl.float64)
        sums += error * error
    tl.store(Scores + row * COUNT + candidate, tl.sum(sums, 0))


@triton.jit
def _pack_best(W, Maxima, Ratios, Scores, Packed, Scales, K: tl.constexpr,
               COUNT: tl.constexpr, PADDED_COUNT: tl.constexpr, BLOCK: tl.constexpr):
    row, tile = tl.program_id(0), tl.program_id(1)
    candidate = tl.arange(0, PADDED_COUNT)
    row_scores = Scores + row * COUNT
    scores = tl.load(row_scores + candidate, candidate < COUNT, other=float("inf"))
    # The first minimum prefers the largest retained range on equal scores.
    best = tl.argmin(scores, axis=0, tie_break_left=True)
    # Keep the canonical baseline when improvement is within FP64 reduction noise.
    best = tl.where(tl.min(scores, 0) < tl.load(row_scores) * (1.0 - 1e-12), best, 0)
    clip = tl.load(Maxima + row) * tl.load(Ratios + best)
    if tile == 0:
        tl.store(Scales + row, _scale(clip))
    j = tile * BLOCK + tl.arange(0, BLOCK)
    even = tl.load(W + row * K + 2 * j, j < K // 2, other=0.0).to(tl.float32)
    odd = tl.load(W + row * K + 2 * j + 1, j < K // 2, other=0.0).to(tl.float32)
    a, b = _abs_code(even, clip), _abs_code(odd, clip)
    a = tl.where(even < 0, -a, a)
    b = tl.where(odd < 0, -b, b)
    byte = ((a & 15) | ((b & 15) << 4)).to(tl.int8)
    tl.store(Packed + row * (K // 2) + j, byte, j < K // 2)


@torch.no_grad()
def quantize_mse(w):
    """Return int8 packed weights [rows, K/2] and one FP32 scale per row.

    Takes a CUDA FP16/BF16 matrix of finite model weights with even positive K.
    Candidate 1.0 reproduces the project's absmax quantizer. Each row selects
    a scale by SSE between the original weights and the FP32 reconstruction.
    """
    if w.ndim != 2 or w.shape[1] == 0 or w.shape[1] % 2:
        raise ValueError("Expected a matrix with positive, even width")
    if not w.is_cuda or w.dtype not in (torch.float16, torch.bfloat16):
        raise ValueError("Expected CUDA FP16 or BF16 weights")
    w = w.contiguous()
    rows, width = w.shape
    packed = torch.empty((rows, width // 2), device=w.device, dtype=torch.int8)
    scales = torch.empty(rows, device=w.device, dtype=torch.float32)
    if rows == 0:
        return packed, scales
    count = len(RATIOS)
    tile = min(512, triton.next_power_of_2(width))
    pack_tile = min(256, triton.next_power_of_2(width // 2))
    chunk = min(rows, 2048)
    # Reuse the scratch on the current CUDA stream, including for the LM head.
    with torch.cuda.device(w.device):
        ratios = torch.tensor(RATIOS, device=w.device, dtype=torch.float32)
        maxima = torch.empty(chunk, device=w.device, dtype=torch.float32)
        scores = torch.empty((chunk, count), device=w.device, dtype=torch.float64)
        for first in range(0, rows, chunk):
            n = min(rows - first, chunk)
            part = w[first:first + n]
            _row_max[(n,)](part, maxima, width, tile)
            _score[(n, count)](part, maxima, ratios, scores, width, count, tile, enable_fp_fusion=False)
            _pack_best[(n, triton.cdiv(width // 2, pack_tile))](
                part, maxima, ratios, scores, packed[first:first + n], scales[first:first + n],
                width, count, triton.next_power_of_2(count), pack_tile)
    return packed, scales


def quantize_fp16_to_int4_packed(W, method="two_pass"):
    """Симметричная per-row FP16/BF16 -> packed INT4 + FP32 scales.

    method="mse" selects the least-error scale from 18 candidates per row.
    "two_pass" (default) and "fused" use the original absmax scale.
    Контракт входа: CUDA, конечные числа, 2D, N чётное.
    Проверка конечности не выполняется внутри, чтобы не добавлять GPU sync.
    """
    assert W.is_cuda and W.ndim == 2
    assert W.dtype in (torch.float16, torch.bfloat16)
    M, N = W.shape
    assert M > 0 and N > 0 and N % 2 == 0
    W = W.contiguous()
    if method == "mse":
        return quantize_mse(W)
    if method not in ("two_pass", "fused"):
        raise ValueError("method must be 'mse', 'two_pass' or 'fused'")
    packed = torch.empty((M, N // 2), device=W.device, dtype=torch.int8)
    scales = torch.empty((M,), device=W.device, dtype=torch.float32)
    if method == "two_pass":
        row_max = torch.empty((M,), device=W.device, dtype=torch.float32)
        row_max_kernel[(M,)](W, row_max, N, BLOCK_N=1024, num_warps=4)
        B = 256
        quantize_pack_kernel[(M, triton.cdiv(N // 2, B))](
            W, row_max, packed, scales, N, B, num_warps=4
        )
    else:
        fused_quantize_pack_kernel[(M,)](
            W, packed, scales, N, BLOCK_N=triton.next_power_of_2(N), num_warps=4
        )
    return packed, scales

Writing task3_quantizer.py


In [3]:
%%writefile task3_x16_matmul.py
"""Inference-only FP16 or BF16 activations @ signed, per-row INT4 weights.T.

Storage contract (shared with the quantization kernel):
    packed: int8 [N, K // 2], K must be even.
    q[n, 2*j] is the low nibble; q[n, 2*j+1] is the high nibble.
    Nibbles use two's complement; the quantizer emits values in [-7, 7].
    scales: float32 [N], W[n, k] = q[n, k] * scales[n].

No full dequantized weight matrix is allocated. Because scales are per-row,
we compute (X @ q.T) * scales in FP32, then cast to the requested output type.
All INT4 integers are exactly representable in both activation types.
"""

import torch
import triton
import triton.language as tl


@triton.jit
def _bf16_int4_matmul_kernel(
    x_ptr,
    packed_ptr,
    scales_ptr,
    y_ptr,
    M,
    N,
    K,
    stride_xm,
    stride_xk,
    stride_pn,
    stride_pk,
    stride_s,
    BLOCK_M: tl.constexpr,
    BLOCK_N: tl.constexpr,
    BLOCK_K: tl.constexpr,
):
    rows = tl.program_id(0) * BLOCK_M + tl.arange(0, BLOCK_M)
    cols = tl.program_id(1) * BLOCK_N + tl.arange(0, BLOCK_N)
    k = tl.arange(0, BLOCK_K)
    packed_k = tl.arange(0, BLOCK_K // 2)
    acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)

    for tile in range(tl.cdiv(K, BLOCK_K)):
        x_k = tile * BLOCK_K + k
        x = tl.load(
            x_ptr + rows[:, None] * stride_xm + x_k[None, :] * stride_xk,
            mask=(rows[:, None] < M) & (x_k[None, :] < K),
            other=0.0,
        )

        # Load every byte once: a row-major [BLOCK_N, BLOCK_K // 2] tile.
        p_k = tile * (BLOCK_K // 2) + packed_k
        packed = tl.load(
            packed_ptr + cols[:, None] * stride_pn + p_k[None, :] * stride_pk,
            mask=(cols[:, None] < N) & (p_k[None, :] < K // 2),
            other=0,
        ).to(tl.int32)

        low = packed & 0xF
        high = (packed >> 4) & 0xF
        # Two's complement: 0..7 stay positive; 8..15 become -8..-1.
        low = (low ^ 8) - 8
        high = (high ^ 8) - 8
        # join adds a final axis [low, high]; reshape interleaves each pair.
        q = tl.reshape(tl.join(low, high), (BLOCK_N, BLOCK_K))
        weights_t = tl.trans(q.to(x.dtype))
        acc = tl.dot(x, weights_t, acc)

    scale = tl.load(scales_ptr + cols * stride_s, mask=cols < N, other=0.0)
    result = acc * scale[None, :]
    tl.store(
        y_ptr + rows[:, None] * N + cols[None, :],
        result,
        mask=(rows[:, None] < M) & (cols[None, :] < N),
    )


def matmul_x16_int4(
    x: torch.Tensor,
    packed: torch.Tensor,
    scales: torch.Tensor,
    *,
    out_dtype: torch.dtype | None = None,
) -> torch.Tensor:
    """Return X @ dequantize(packed, scales).T on the current CUDA stream.

    Args:
        x: FP16 or BF16 CUDA tensor [M, K].
        packed: INT8 CUDA tensor [N, K // 2], low nibble first.
        scales: FP32 CUDA tensor [N], one finite scale per weight row.
        out_dtype: x.dtype (default) or FP32, useful for numerical validation.

    Strided input views are supported; output is contiguous [M, N]. K must
    be even, matching the quantizer's format. This function is for inference
    and does not implement autograd. On NVIDIA T4, use FP16 activations.
    NVIDIA Ampere or newer supports both FP16 and BF16 dot products.
    Per-row scales are applied after FP32 accumulation.
    """
    for name, tensor in (("x", x), ("packed", packed), ("scales", scales)):
        if not isinstance(tensor, torch.Tensor):
            raise TypeError(f"{name} must be a torch.Tensor")
    if x.ndim != 2 or packed.ndim != 2 or scales.ndim != 1:
        raise ValueError("Expected x [M, K], packed [N, K // 2], scales [N]")
    if x.dtype not in (torch.float16, torch.bfloat16):
        raise TypeError("x must have dtype torch.float16 or torch.bfloat16")
    if packed.dtype != torch.int8:
        raise TypeError("packed must have dtype torch.int8")
    if scales.dtype != torch.float32:
        raise TypeError("scales must have dtype torch.float32")
    if out_dtype is None:
        out_dtype = x.dtype
    if out_dtype not in (x.dtype, torch.float32):
        raise TypeError("out_dtype must be x.dtype or torch.float32")

    m, k = x.shape
    n = packed.shape[0]
    if k % 2:
        raise ValueError("K (in_features) must be even for two INT4 weights per byte")
    if packed.shape[1] != k // 2:
        raise ValueError("packed.shape must be [N, x.shape[1] // 2]")
    if scales.shape[0] != n:
        raise ValueError("scales must contain exactly one scale per weight row")
    if not (x.is_cuda and packed.is_cuda and scales.is_cuda):
        raise ValueError("x, packed and scales must all be CUDA tensors")
    if x.device != packed.device or x.device != scales.device:
        raise ValueError("x, packed and scales must be on the same CUDA device")

    y = torch.empty((m, n), dtype=out_dtype, device=x.device)
    if m == 0 or n == 0:
        return y
    if k == 0:
        return y.zero_()

    block_m = 16 if m <= 16 else 32 if m <= 32 else 64
    block_n, block_k = 64, 64
    grid = (triton.cdiv(m, block_m), triton.cdiv(n, block_n))
    with torch.cuda.device(x.device):
        _bf16_int4_matmul_kernel[grid](
            x, packed, scales, y,
            m, n, k,
            x.stride(0), x.stride(1),
            packed.stride(0), packed.stride(1), scales.stride(0),
            BLOCK_M=block_m,
            BLOCK_N=block_n,
            BLOCK_K=block_k,
            num_warps=4,
            num_stages=3 if torch.cuda.get_device_capability(x.device)[0] >= 8 else 1,
        )
    return y


def matmul_bf16_int4(
    x: torch.Tensor,
    packed: torch.Tensor,
    scales: torch.Tensor,
    *,
    out_dtype: torch.dtype = torch.bfloat16,
) -> torch.Tensor:
    """Multiply BF16 activations by packed INT4 weights with a BF16 or FP32 output."""
    if not isinstance(x, torch.Tensor):
        raise TypeError("x must be a torch.Tensor")
    if x.dtype != torch.bfloat16:
        raise TypeError("x must have dtype torch.bfloat16")
    if out_dtype not in (torch.bfloat16, torch.float32):
        raise TypeError("out_dtype must be torch.bfloat16 or torch.float32")
    return matmul_x16_int4(x, packed, scales, out_dtype=out_dtype)

Writing task3_x16_matmul.py


In [4]:
%%writefile task3_mma75.py
"""Turing FP16 tensor-core probe for contiguous, aligned benchmark shapes."""
import torch
import triton
import triton.language as tl


@triton.jit
def _mma16x8(a0, a1, b, c0, c1, c2, c3):
    return tl.inline_asm_elementwise(
        asm="mma.sync.aligned.m16n8k8.row.col.f32.f16.f16.f32 {$0,$1,$2,$3}, {$4,$5}, {$6}, {$7,$8,$9,$10};",
        constraints="=f,=f,=f,=f,r,r,r,f,f,f,f",
        args=[a0, a1, b, c0, c1, c2, c3],
        dtype=(tl.float32, tl.float32, tl.float32, tl.float32),
        is_pure=True, pack=1,
    )


@triton.jit
def _packed_mma75(X, P, S, Y, M: tl.constexpr, N: tl.constexpr, K: tl.constexpr):
    # Each of four warps computes a 16-by-8 output fragment using the PTX
    # lane-to-fragment mapping. Four neighboring lanes own each 8-element K slice.
    tid = tl.inline_asm_elementwise(
        "mov.u32 $0,%tid.x;", "=r,r", [tl.arange(0, 128)],
        dtype=tl.int32, is_pure=True, pack=1,
    )
    warp = tid // 32
    lane = tid % 32
    m = tl.program_id(0) * 32 + (warp // 2) * 16
    n = tl.program_id(1) * 16 + (warp % 2) * 8
    group = lane // 4
    pair = lane % 4
    xpair = X.to(tl.pointer_type(tl.uint32))
    a_ptr = xpair + (m + group) * (K // 2) + pair
    b_ptr = P + (n + group) * (K // 2) + pair
    c0 = tl.full((128,), 0, tl.float32)
    c1 = tl.full((128,), 0, tl.float32)
    c2 = tl.full((128,), 0, tl.float32)
    c3 = tl.full((128,), 0, tl.float32)
    for tile in range(K // 32):
        for step in tl.static_range(4):
            off = tile * 16 + step * 4
            a0 = tl.load(a_ptr + off)
            a1 = tl.load(a_ptr + off + 8 * (K // 2))
            p = tl.load(b_ptr + off).to(tl.uint8).to(tl.int32)
            lo = (((p & 15) ^ 8) - 8).to(tl.float16).to(tl.uint16, bitcast=True)
            hi = ((p >> 4 ^ 8) - 8).to(tl.float16).to(tl.uint16, bitcast=True)
            b = lo.to(tl.uint32) | (hi.to(tl.uint32) << 16)
            c0, c1, c2, c3 = _mma16x8(a0, a1, b, c0, c1, c2, c3)
    col = n + pair * 2
    out = Y + (m + group) * N + col
    scale0 = tl.load(S + col)
    scale1 = tl.load(S + col + 1)
    tl.store(out, c0 * scale0)
    tl.store(out + 1, c1 * scale1)
    tl.store(out + 8 * N, c2 * scale0)
    tl.store(out + 8 * N + 1, c3 * scale1)


def matmul_fp16_int4_mma75(x, packed, scales):
    """SM75 diagnostic for the multiples-of-32/16/32 shapes used in the benchmark."""
    if x.ndim != 2 or packed.ndim != 2 or scales.ndim != 1:
        raise ValueError("Expected x[M,K], packed[N,K/2] and scales[N].")
    if not x.is_cuda or packed.device != x.device or scales.device != x.device:
        raise ValueError("All inputs must be on one CUDA device.")
    if x.dtype != torch.float16 or packed.dtype != torch.int8 or scales.dtype != torch.float32:
        raise TypeError("Expected FP16 activations, INT8 packed weights and FP32 scales.")
    m, k = x.shape
    n = packed.size(0)
    if min(m, n, k) <= 0 or m % 32 or n % 16 or k % 32:
        raise ValueError("This diagnostic kernel requires M, N, K divisible by 32, 16, 32 respectively.")
    if packed.size(1) != k // 2 or scales.size(0) != n:
        raise ValueError("Incompatible packed weight or scale shape.")
    if not all(t.is_contiguous() for t in (x, packed, scales)) or x.data_ptr() % 4:
        raise ValueError("Expected contiguous tensors and a 4-byte aligned activation pointer.")
    y = torch.empty((m, n), device=x.device, dtype=x.dtype)
    with torch.cuda.device(x.device):
        _packed_mma75[(m // 32, n // 16)](x, packed, scales, y, m, n, k, num_warps=4, num_stages=1)
    return y

Writing task3_mma75.py


In [5]:
from task3_quantizer import quantize_fp16_to_int4_packed
from task3_x16_matmul import matmul_x16_int4

T4_KERNEL = "mma.sync"
if T4_KERNEL not in ("mma.sync", "tl.dot"):
    raise ValueError('Для T4_KERNEL укажите "mma.sync" или "tl.dot".')
use_mma = (device.major, device.minor) == (7, 5) and T4_KERNEL == "mma.sync"
if use_mma:
    from task3_mma75 import matmul_fp16_int4_mma75 as run_int4
else:
    run_int4 = matmul_x16_int4
kernel_name = "mma.sync" if use_mma else "tl.dot"
print(f"INT4: {kernel_name} · {dtype_label}")

# При масштабе 2 значения ±1 лежат ровно посередине между соседними уровнями.
example = torch.tensor([[14, -14, 1, -1, 0, 6, -8, 12], [0] * 8],
                       device="cuda", dtype=torch.float16)
packed, scales = quantize_fp16_to_int4_packed(example, method="two_pass")
assert packed.dtype == torch.int8 and scales.dtype == torch.float32
assert packed.to(torch.uint8).tolist() == [[0x97, 0xF1, 0x30, 0x6C], [0] * 4]
assert scales.tolist() == [2.0, 1.0]

mse_packed, mse_scales = quantize_fp16_to_int4_packed(example, method="mse")
assert mse_packed.shape == packed.shape and mse_packed.dtype == packed.dtype
assert mse_scales.shape == scales.shape and mse_scales.dtype == scales.dtype
assert mse_scales[1].item() == 1.0 and not mse_packed[1].any()
print("Проверены формат упаковки, порядок полубайтов и нулевая строка.")

INT4: mma.sync · FP16


Проверены формат упаковки, порядок полубайтов и нулевая строка.


## Что входит в замер

Для каждого размера NumPy генерирует матрицу весов в FP16. Плотная версия получает эти веса в типе активаций. Обе INT4-версии получают свои упакованные копии. На вход всем трём подаём одинаковые активации. На T4 генератор воспроизводит входы из локального замера на GTX.

Перед замером проверяем 256 элементов результата. Для каждого считаем скалярное произведение по всем K признакам в FP64 на CPU. У каждого INT4-варианта свой эталон — берём именно его байты и масштаб, чтобы проверить само умножение.

Каждый метод запускаем четыре раза для прогрева. Затем перебираем все шесть порядков запуска трёх методов, чтобы каждый побывал и первым, и последним. Для самой большой матрицы при 2048 токенах это шесть повторов на метод. Для остальных проходим те же порядки ещё раз в обратной последовательности.

CUDA-события ставим вокруг вызова умножения. К этому моменту веса уже сжаты, данные на GPU, кернели скомпилированы. Каждая функция создаёт выходной тензор при вызове. В архивном замере на GTX выход выделяли заранее, а алгоритм плотного умножения выбирали явно через cuBLASLt.

In [6]:
from hashlib import sha256
from itertools import permutations
from statistics import median

import numpy as np


PROJECTIONS = (
    ("q_proj / o_proj", 2048, 2048),
    ("k_proj / v_proj", 512, 2048),
    ("gate_proj / up_proj", 8192, 2048),
    ("down_proj", 2048, 8192),
    ("lm_head", 128256, 2048),
)
TOKEN_COUNTS = (128, 512, 2048)
SEED, WARMUPS = 42, 4
# При умножении в PyTorch оставляем промежуточные суммы в FP32, в том числе для split-K.
torch.backends.cuda.matmul.allow_fp16_reduced_precision_reduction = False
torch.backends.cuda.matmul.allow_bf16_reduced_precision_reduction = False


def make_weights(n, k, rng):
    w = torch.empty((n, k), dtype=torch.float16, device="cuda")
    digest = sha256()
    for start in range(0, n, 1024):
        size = min(1024, n - start)
        part = rng.standard_normal((size, k), dtype=np.float32).astype(np.float16)
        digest.update(memoryview(part).cast("B"))
        w[start:start + size].copy_(torch.from_numpy(part))
    return w, digest.hexdigest()


def decoded_rows(packed, scales, cols):
    byte = packed[cols].cpu().numpy().view(np.uint8)
    codes = np.stack((byte & 15, byte >> 4), axis=-1).reshape(byte.shape[0], -1)
    signed = (codes.astype(np.int16) ^ 8) - 8
    return signed.astype(np.float64) * scales[cols].cpu().numpy()[:, None]


def time_methods(methods, short):
    for _ in range(WARMUPS):
        for fn in methods.values():
            fn()
            torch.cuda.synchronize()

    orders = list(permutations(methods))
    if not short:
        orders += orders[::-1]
    times = {name: [] for name in methods}
    start, end = torch.cuda.Event(enable_timing=True), torch.cuda.Event(enable_timing=True)
    for order in orders:
        for name in order:
            start.record()
            methods[name]()
            end.record()
            end.synchronize()
            times[name].append(start.elapsed_time(end))
    return times, orders


@torch.inference_mode()
def run_benchmark():
    summary, raw = [], []
    for index, (label, n, k) in enumerate(PROJECTIONS):
        rng = np.random.default_rng(np.random.SeedSequence([SEED, index]))
        w, weight_sha = make_weights(n, k, rng)
        absmax = quantize_fp16_to_int4_packed(w, method="two_pass")
        mse = quantize_fp16_to_int4_packed(w, method="mse")
        wt = w.to(x_dtype).T
        cols = np.linspace(0, n - 1, 16).round().astype(np.int64)
        c = torch.from_numpy(cols).to(w.device)
        refs = {
            "dense": wt.T[c].double().cpu().numpy(),
            "absmax": decoded_rows(*absmax, c),
            "mse": decoded_rows(*mse, c),
        }
        del w
        for m in TOKEN_COUNTS:
            values = rng.standard_normal((m, k), dtype=np.float32).astype(np.float16)
            x_sha = sha256(memoryview(values).cast("B")).hexdigest()
            x = torch.from_numpy(values).to(device="cuda", dtype=x_dtype)
            del values
            methods = {
                "dense": lambda: torch.mm(x, wt),
                "absmax": lambda: run_int4(x, *absmax),
                "mse": lambda: run_int4(x, *mse),
            }
            rows = torch.linspace(0, m - 1, 16, dtype=torch.float64, device=x.device).round().long()
            sampled_x = x[rows].double().cpu().numpy()
            errors = {}
            for name, fn in methods.items():
                expected = sampled_x @ refs[name].T
                actual = fn()[rows[:, None], c].double().cpu().numpy()
                np.testing.assert_allclose(actual, expected, rtol=0.005, atol=0.01)
                errors[name] = float(np.max(np.abs(actual - expected)))
            times, orders = time_methods(methods, short=(n, m) == (128256, 2048))
            dense, old, new = (median(times[key]) for key in ("dense", "absmax", "mse"))
            record = {
                "projection": label, "M": m, "N": n, "K": k, "dtype": dtype_label,
                "int4_kernel": kernel_name, dense_column: dense, "int4_ms": old,
                "speedup": dense / old, "absmax_ms": old, "mse_ms": new,
                "mse_speedup": dense / new, "mse_vs_absmax_pct": 100 * (new / old - 1),
                "trials_per_method": len(orders),
            }
            for name, measurements in times.items():
                record[name + "_p25_ms"], record[name + "_p75_ms"] = np.percentile(measurements, [25, 75]).tolist()
            summary.append(record)
            raw.append({**record, "samples_ms": times, "orders": orders, "max_abs_error": errors,
                        "checked_per_method": 256, "weight_sha256": weight_sha,
                        "activation_fp16_source_sha256": x_sha})
            print(f"{len(summary):02}/15 {label:20} M={m:4}   {dtype_label} {dense:.3f} мс   "
                  f"absmax {old:.3f} мс   MSE {new:.3f} мс", flush=True)
        del x, wt, absmax, mse
    return summary, raw

In [7]:
results, trials = run_benchmark()

01/15 q_proj / o_proj      M= 128   FP16 2.314 мс   absmax 1.359 мс   MSE 1.359 мс


02/15 q_proj / o_proj      M= 512   FP16 7.651 мс   absmax 4.072 мс   MSE 4.074 мс


03/15 q_proj / o_proj      M=2048   FP16 28.051 мс   absmax 16.202 мс   MSE 16.204 мс


04/15 k_proj / v_proj      M= 128   FP16 0.494 мс   absmax 0.291 мс   MSE 0.291 мс


05/15 k_proj / v_proj      M= 512   FP16 2.555 мс   absmax 1.051 мс   MSE 1.051 мс


06/15 k_proj / v_proj      M=2048   FP16 7.014 мс   absmax 4.092 мс   MSE 4.092 мс


... (5 outputs left out)

12/15 down_proj            M=2048   FP16 113.122 мс   absmax 65.399 мс   MSE 65.386 мс


13/15 lm_head              M= 128   FP16 107.961 мс   absmax 63.821 мс   MSE 63.820 мс


14/15 lm_head              M= 512   FP16 434.809 мс   absmax 259.600 мс   MSE 259.295 мс


15/15 lm_head              M=2048   FP16 1745.781 мс   absmax 1064.735 мс   MSE 1064.422 мс


## Как читать результаты

В столбцах `absmax_ms` и `mse_ms` показано медианное время одного умножения в миллисекундах. `mse_vs_absmax_pct` показывает, как изменилось время при переходе на MSE. Отрицательное число означает, что MSE-вариант отработал быстрее. Если разница мала, посмотрите на разброс повторов.

Чтобы сравнить MSE с плотным умножением, смотрите на `mse_speedup`. Это время `torch.mm`, делённое на время MSE-варианта. Значение выше 1 означает выигрыш INT4. В столбцах `int4_ms` и `speedup` по-прежнему лежат результаты для `absmax`, как в предыдущем CSV.

В панели **Файлы** Colab появятся таблица CSV и JSON с отдельными повторами и результатами проверок.

In [8]:
import csv
import json
from pathlib import Path

metadata = {**runtime, "seed": SEED, "numpy_version": np.__version__, "warmups_per_method": WARMUPS,
            "dense_method": "torch.mm", "quantizers": "two_pass,mse", "dtype": dtype_label,
            "int4_kernel": kernel_name}
stem = "llama_int4_benchmark_t4_mma" if use_mma else "llama_int4_benchmark"
csv_records = [{**row, **metadata} for row in results]
with open(stem + ".csv", "w", encoding="utf8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=list(csv_records[0]))
    writer.writeheader()
    writer.writerows(csv_records)
Path(stem + "_raw.json").write_text(
    json.dumps({"meta": metadata, "rows": trials}, ensure_ascii=False, indent=2, allow_nan=False),
    encoding="utf8",
)
print(stem + ".csv", stem + "_raw.json")

from IPython.display import Markdown, display

lines = [
    f"| Проекция | Токены | {dtype_label}, мс | INT4 absmax, мс | INT4 MSE, мс | Ускорение MSE к плотному |",
    "|---|---:|---:|---:|---:|---:|",
]
for row in results:
    lines.append(
        f"| {row['projection']} | {row['M']} | {row[dense_column]:.3f} | "
        f"{row['absmax_ms']:.3f} | {row['mse_ms']:.3f} | {row['mse_speedup']:.2f}× |"
    )
display(Markdown("\n".join(lines)))


llama_int4_benchmark_t4_mma.csv llama_int4_benchmark_t4_mma_raw.json


| Проекция | Токены | FP16, мс | INT4 absmax, мс | INT4 MSE, мс | Ускорение MSE к плотному |
|---|---:|---:|---:|---:|---:|
| q_proj / o_proj | 128 | 2.314 | 1.359 | 1.359 | 1.70× |
| q_proj / o_proj | 512 | 7.651 | 4.072 | 4.074 | 1.88× |
| q_proj / o_proj | 2048 | 28.051 | 16.202 | 16.204 | 1.73× |
| k_proj / v_proj | 128 | 0.494 | 0.291 | 0.291 | 1.70× |
| k_proj / v_proj | 512 | 2.555 | 1.051 | 1.051 | 2.43× |
| k_proj / v_proj | 2048 | 7.014 | 4.092 | 4.092 | 1.71× |
| gate_proj / up_proj | 128 | 6.849 | 4.080 | 4.080 | 1.68× |
| gate_proj / up_proj | 512 | 28.016 | 16.202 | 16.201 | 1.73× |
| gate_proj / up_proj | 2048 | 110.516 | 65.278 | 65.122 | 1.70× |
| down_proj | 128 | 10.194 | 4.180 | 4.177 | 2.44× |
| down_proj | 512 | 30.562 | 16.296 | 16.295 | 1.88× |
| down_proj | 2048 | 113.122 | 65.399 | 65.386 | 1.73× |
| lm_head | 128 | 107.961 | 63.821 | 63.820 | 1.69× |
| lm_head | 512 | 434.809 | 259.600 | 259.295 | 1.68× |
| lm_head | 2048 | 1745.781 | 1064.735 | 1064.422 | 1.64× |